# Spatial Model Performance

Compare ridge, random forest, histogram gradient boosting and training-mean baselines
within one explicitly selected experiment. Metrics, predictions and model diagnostics
all come from that run's manifest. Historical RF files remain in their original folder.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "pipeline" / "config").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from pipeline.modeling import load_run
from pipeline.utils.paths import load_paths, path_value

PATHS = load_paths()
RUN_ROOT = path_value(PATHS, "external_data_root") / "outputs/epvi_prediction/runs"
pd.set_option("display.max_columns", 100)

In [ ]:
# Copy the exact ID printed by 3_1_model_training.ipynb; no implicit latest-file lookup.
RUN_ID = "REPLACE_WITH_RUN_ID"
if RUN_ID == "REPLACE_WITH_RUN_ID":
    available = sorted(p.parent.name for p in RUN_ROOT.glob("*/manifest.json"))
    raise ValueError(f"Set RUN_ID to an experiment directory. Available runs: {available}")
results = load_run(RUN_ROOT / RUN_ID)
print("Scope:", results.manifest["config"]["scope"])
print("Tuning:", results.manifest["config"]["evaluation"])
print(results.manifest["oof_note"])

## Fixed Test Comparison

Compare models within targets, including the training-mean reference. RMSE and MAE
retain each target's units; do not average them across targets. These results describe
the frozen holdout, whose prior inspection/replacement is documented in the tuning history.

In [ ]:
display(results.comparison_table())
test_metrics = results.metrics.loc[results.metrics.prediction_role.eq("fixed_test")]
for target, frame in test_metrics.groupby("target", sort=False):
    table = frame.pivot(index="model", columns="metric", values="value")
    columns = [c for c in ["rmse", "r2", "spearman"] if c in table]
    fig, axes = plt.subplots(1, len(columns), figsize=(5 * len(columns), 4), squeeze=False)
    for ax, metric in zip(axes.flat, columns):
        table[metric].plot.barh(ax=ax)
        ax.set_title(f"{target}: {metric}")
        ax.axvline(0, color="black", linewidth=0.7)
        ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()
display(results.metrics.loc[results.metrics.undefined_reason.notna()])

## Training Spatial CV Diagnostics

The table below uses pooled OOF predictions with globally selected hyperparameters;
it is not an independent evaluation of tuning. Per-fold scores are also retained.
Their arithmetic mean (especially R²) is not the same as the pooled OOF metric.

In [ ]:
display(results.comparison_table("train_spatial_cv"))
display(results.fold_metrics.groupby(["target", "model", "metric"]).value
        .agg(["mean", "std", "count"]).rename(columns={"count": "defined_folds"}))
display(results.folds)

## Predictions and Residuals for a Selected Model

Change `MODEL` to inspect another model or the `training_mean` baseline.
Residuals are observed minus predicted. Regional metrics help show spatial variation
that an overall score hides; undefined small/constant-region metrics retain reasons.

In [ ]:
MODEL = "random_forest"
if MODEL not in set(results.predictions.model):
    raise ValueError(f"Model not in this run: {MODEL}")
test_predictions = results.predictions.loc[
    results.predictions.model.eq(MODEL) & results.predictions.prediction_role.eq("fixed_test")]
for target, frame in test_predictions.groupby("target", sort=False):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].scatter(frame.observed, frame.predicted, alpha=0.6, s=18)
    lo = min(frame.observed.min(), frame.predicted.min())
    hi = max(frame.observed.max(), frame.predicted.max())
    axes[0].plot([lo, hi], [lo, hi], color="black", linewidth=0.8)
    axes[0].set(xlabel="Observed", ylabel="Predicted", title=f"{target}: {MODEL}")
    axes[1].hist(frame.residual, bins=30)
    axes[1].axvline(0, color="black", linewidth=0.8)
    axes[1].set(xlabel="Observed - predicted", ylabel="Count", title="Fixed test residuals")
    plt.tight_layout()
    plt.show()
display(results.regional_metrics.loc[results.regional_metrics.model.eq(MODEL)])
display(test_predictions.assign(abs_residual=test_predictions.residual.abs())
        .sort_values("abs_residual", ascending=False).groupby("target").head(15))

## Model-Specific Diagnostics and Selected Parameters

Ridge coefficients describe a one-standard-deviation change in each imputed predictor;
random forest impurity importances describe training-tree impurity reductions. These
are different quantities and should not be compared numerically. Histogram boosting
has no built-in diagnostic exported here. No test-based feature selection is performed.

In [ ]:
diagnostics = results.diagnostics.loc[results.diagnostics.model.eq(MODEL)]
if diagnostics.empty:
    print("No model-specific feature diagnostic for", MODEL)
for (target, diagnostic), frame in diagnostics.groupby(["target", "diagnostic"], sort=False):
    top = frame.assign(magnitude=frame.value.abs()).nlargest(15, "magnitude").sort_values("value")
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(top.feature, top.value)
    ax.set_title(f"{target}: {diagnostic}")
    plt.tight_layout()
    plt.show()

fit_details = []
for pair in results.manifest["pairs"]:
    if pair["model"] == MODEL:
        fit_details.append(json.loads((results.run_dir / pair["directory"] / "fit.json").read_text(encoding="utf-8")))
display(pd.DataFrame(fit_details))